# Import Libraries

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import copy

# Data Loading and Initial Exploration

## 1. Load Dataset

In [2]:
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_PATH = PROJECT_ROOT / "data" / "audi-etron-data.parquet"

print(f"Project Root: {PROJECT_ROOT}") # verify the project directory path
print(f"Data Path: {DATA_PATH}")       # verify the data path

df = pd.read_parquet(DATA_PATH)

Project Root: C:\Users\mayur\OneDrive - University of Oklahoma\DataScience\ev-battery-anomaly-detection
Data Path: C:\Users\mayur\OneDrive - University of Oklahoma\DataScience\ev-battery-anomaly-detection\data\audi-etron-data.parquet


## 2. Understand Data Structure

In [3]:
print(f"Shape(rows x columns): {df.shape}")              # shape i.e. rows x columns
print(f"Column names: {df.columns.tolist()}")            # column names

Shape(rows x columns): (10534347, 5)
Column names: ['Timestamp', 'VIN', 'SOC', 'Mileage', 'Home']


In [4]:
print(df.info())         # Information about the data structure

<class 'pandas.DataFrame'>
RangeIndex: 10534347 entries, 0 to 10534346
Data columns (total 5 columns):
 #   Column     Dtype         
---  ------     -----         
 0   Timestamp  datetime64[us]
 1   VIN        str           
 2   SOC        int64         
 3   Mileage    int64         
 4   Home       int64         
dtypes: datetime64[us](1), int64(3), str(1)
memory usage: 843.9 MB
None


In [5]:
df.dtypes               # Data types for each column object

Timestamp    datetime64[us]
VIN                     str
SOC                   int64
Mileage               int64
Home                  int64
dtype: object

### About the features/columns

1) Timestamp - Time and date of when the measurement was recorded
2) VIN - Anonymized, hash/encoded VIN number for privacy protection. Same VIN implies same car
3) SOC - State of charge i.e. battery percentage from 0 to 100. 0 implies 0% battery and 100 implies fully charged, 100% battery
4) Mileage - Cumulative odometer reading, should always increase with time
5) Home - Binary indicator to state if vehicle is at home (1) or away (0). Away implies driving, charging or parked elsewhere

## 3. Initial Data Exploration

In [6]:
df.head()                              # first five rows

,Timestamp,VIN,SOC,Mileage,Home
0,2020-02-09 22:28:14,+2VqEDzfPo3GYxjixcXwaegwlO/gn1Rnp3hJLC5D62Q=,99,98,0
1,2020-02-09 22:28:35,+2VqEDzfPo3GYxjixcXwaegwlO/gn1Rnp3hJLC5D62Q=,99,98,0
2,2020-02-09 22:30:22,+2VqEDzfPo3GYxjixcXwaegwlO/gn1Rnp3hJLC5D62Q=,99,98,0
3,2020-02-09 22:31:58,+2VqEDzfPo3GYxjixcXwaegwlO/gn1Rnp3hJLC5D62Q=,99,98,0
4,2020-02-09 22:32:47,+2VqEDzfPo3GYxjixcXwaegwlO/gn1Rnp3hJLC5D62Q=,99,98,0


In [7]:
df.describe()                          # summary statistics for numerical features

,Timestamp,SOC,Mileage,Home
count,10534347,1.053435e+07,1.053435e+07,1.053435e+07
mean,2020-01-29 00:40:12.130589,7.062343e+01,7.658934e+03,3.463527e-01
min,2019-03-19 20:39:10,0.000000e+00,1.300000e+01,0.000000e+00
25%,2019-10-24 03:46:10,5.700000e+01,2.799000e+03,0.000000e+00
50%,2020-01-20 17:17:55,7.400000e+01,5.977000e+03,0.000000e+00
75%,2020-04-25 17:20:05,8.800000e+01,1.054500e+04,1.000000e+00
max,2022-04-07 14:21:57,1.000000e+02,1.067860e+05,1.000000e+00
std,NaN,2.125142e+01,6.843549e+03,4.758073e-01


## 4. Check Data Quality

In [8]:
# check for missing values
df.isna().sum()

Timestamp    0
VIN          0
SOC          0
Mileage      0
Home         0
dtype: int64

In [9]:
# Unique vehicles
unique_vehicles = len(df['VIN'].unique())
print(f"There are {unique_vehicles} unique vehicles among the {df.shape[0]} vehicles entries")

There are 3082 unique vehicles among the 10534347 vehicles entries


In [10]:
# check for duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"There are {duplicate_rows} dupliate rows out of {df.shape[0]}")

There are 790001 dupliate rows out of 10534347


In [11]:
# Look at duplicated rows
df[df.duplicated()].head(10)

,Timestamp,VIN,SOC,Mileage,Home
28283,2019-08-14 21:18:46,/Xgot4w31aIw13XNxZWDlSoV+1F47PVr7UDqmvivon0=,97,17,0
28284,2019-08-14 21:18:46,/Xgot4w31aIw13XNxZWDlSoV+1F47PVr7UDqmvivon0=,97,17,0
28286,2019-08-14 21:19:08,/Xgot4w31aIw13XNxZWDlSoV+1F47PVr7UDqmvivon0=,97,17,0
28287,2019-08-14 21:19:08,/Xgot4w31aIw13XNxZWDlSoV+1F47PVr7UDqmvivon0=,97,17,0
28289,2019-08-14 21:19:43,/Xgot4w31aIw13XNxZWDlSoV+1F47PVr7UDqmvivon0=,97,17,0
28290,2019-08-14 21:19:43,/Xgot4w31aIw13XNxZWDlSoV+1F47PVr7UDqmvivon0=,97,17,0
28292,2019-08-14 21:43:16,/Xgot4w31aIw13XNxZWDlSoV+1F47PVr7UDqmvivon0=,97,17,0
28293,2019-08-14 21:43:16,/Xgot4w31aIw13XNxZWDlSoV+1F47PVr7UDqmvivon0=,97,17,0
28295,2019-08-14 21:43:33,/Xgot4w31aIw13XNxZWDlSoV+1F47PVr7UDqmvivon0=,97,17,0
28296,2019-08-14 21:43:33,/Xgot4w31aIw13XNxZWDlSoV+1F47PVr7UDqmvivon0=,97,17,0


In [12]:
# Look at duplicated rows
df[df.duplicated()].tail(10)

,Timestamp,VIN,SOC,Mileage,Home
10519296,2020-09-22 15:32:56,yPJbl/Jw5vwGSJ0UeGQcJXUMY33U2FEjta+xYKJv23w=,96,6544,0
10519298,2020-09-22 15:33:30,yPJbl/Jw5vwGSJ0UeGQcJXUMY33U2FEjta+xYKJv23w=,96,6544,0
10519300,2020-09-22 15:34:51,yPJbl/Jw5vwGSJ0UeGQcJXUMY33U2FEjta+xYKJv23w=,96,6544,0
10519302,2020-09-22 15:35:34,yPJbl/Jw5vwGSJ0UeGQcJXUMY33U2FEjta+xYKJv23w=,96,6544,0
10519304,2020-09-22 15:48:15,yPJbl/Jw5vwGSJ0UeGQcJXUMY33U2FEjta+xYKJv23w=,96,6544,0
10519306,2020-09-22 16:11:54,yPJbl/Jw5vwGSJ0UeGQcJXUMY33U2FEjta+xYKJv23w=,96,6544,0
10519308,2020-09-22 16:15:30,yPJbl/Jw5vwGSJ0UeGQcJXUMY33U2FEjta+xYKJv23w=,96,6544,0
10519310,2020-09-22 16:16:42,yPJbl/Jw5vwGSJ0UeGQcJXUMY33U2FEjta+xYKJv23w=,96,6544,0
10519312,2020-10-06 17:28:14,yPJbl/Jw5vwGSJ0UeGQcJXUMY33U2FEjta+xYKJv23w=,92,6546,0
10519314,2020-11-03 19:29:32,yPJbl/Jw5vwGSJ0UeGQcJXUMY33U2FEjta+xYKJv23w=,91,6546,0


### Duplicate rows or not?

Some of the data is actually duplicate with exact date and time, but some duplicates are not actual duplicates. 
They have same date and other values, but their times are different, stating the idle state of vehicle.

## 5. Sanity Check - Track Single Vehicle

In [13]:
# Pick one vehicle to see if the mileage and SOC changes with time
sample_vin = df['VIN'].iloc[0]
sample_data = df[df['VIN'] == sample_vin][['Timestamp','Mileage', 'SOC']]

print(sample_data.head(15))

             Timestamp  Mileage  SOC
0  2020-02-09 22:28:14       98   99
1  2020-02-09 22:28:35       98   99
2  2020-02-09 22:30:22       98   99
3  2020-02-09 22:31:58       98   99
4  2020-02-09 22:32:47       98   99
5  2020-02-09 22:33:16       98   99
6  2020-02-09 23:04:36      136   87
7  2020-02-09 23:04:59      136   87
8  2020-02-09 23:05:09      136   87
9  2020-02-09 23:05:35      136   87
10 2020-02-09 23:17:12      136   87
11 2020-02-09 23:18:11      136   87
12 2020-02-09 23:18:23      136   87
13 2020-02-09 23:19:06      136   87
14 2020-02-09 23:39:04      147   86


### SOC vs Mileage

General trend is when mileage increases then SOC decreases, which makes sense. As the vehicle is driven, the battery charge should deplete over time.

## 6. Key Findings

1. The VIN numbers are anonymized for privacy. There are a total of 3082 unique vehicles.
2. There are no missing values.
3. Initial check reveals duplicate rows. However, not all are duplicate rows. Some jsut correspond to data at different times of a day, when the vehicle was in idle/parked condition